# Data Preparation — Datathon FIAP (Passos Mágicos)

Este notebook realiza a preparação da base de dados das planilhas PEDE2022, PEDE2023 e PEDE2024.

Objetivos:
- padronizar nomes de colunas
- consolidar as três planilhas em uma base única
- corrigir problemas de tipagem
- tratar valores especiais
- padronizar variáveis categóricas
- criar variáveis derivadas de fase, defasagem e IAN
- salvar a base final tratada para uso no EDA, dashboard e modelagem



In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
pd.set_option('display.max_columns', None)

In [ ]:
url = "https://raw.githubusercontent.com/RickPardono/Datathon_FIAP/refs/heads/main/data/raw/BASE_DE_DADOS_PEDE.xlsx"
df_2022 = pd.read_excel(url, sheet_name="PEDE2022")
df_2023 = pd.read_excel(url, sheet_name="PEDE2023")
df_2024 = pd.read_excel(url, sheet_name="PEDE2024")

In [ ]:
print("PEDE2022:", df_2022.shape)
print("PEDE2023:", df_2023.shape)
print("PEDE2024:", df_2024.shape)

PEDE2022: (860, 42)
PEDE2023: (1014, 48)
PEDE2024: (1156, 50)


In [ ]:
df_2022["ano"] = 2022
df_2023["ano"] = 2023
df_2024["ano"] = 2024

In [ ]:
mapping = {
    "Nome": "nome",
    "Nome Anonimizado": "nome",
    "Ano nasc": "ano_nascimento",
    "Data de Nasc": "data_nascimento",
    "Idade 22": "idade",
    "Idade": "idade",
    "Gênero": "genero",
    "Ano ingresso": "ano_ingresso",
    "Instituição de ensino": "instituicao_ensino",
    "Fase ideal": "fase_ideal",
    "Fase Ideal": "fase_ideal",
    "Defas": "defasagem",
    "Defasagem": "defasagem",
    "IAA": "iaa",
    "IEG": "ieg",
    "IPS": "ips",
    "IPP": "ipp",
    "IDA": "ida",
    "IPV": "ipv",
    "IAN": "ian",
    "Matem": "mat",
    "Mat": "mat",
    "Portug": "por",
    "Por": "por",
    "Inglês": "ing",
    "Ing": "ing",
    "Nº Av": "num_avaliacoes",
}

In [ ]:
def padronizar_colunas(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    df = df.rename(columns=mapping).copy()

    if ano == 2022:
        if "INDE 22" in df.columns:
            df = df.rename(columns={"INDE 22": "inde"})
        if "Pedra 22" in df.columns:
            df = df.rename(columns={"Pedra 22": "pedra"})

    elif ano == 2023:
        if "INDE 2023" in df.columns:
            df = df.rename(columns={"INDE 2023": "inde"})
        if "Pedra 2023" in df.columns:
            df = df.rename(columns={"Pedra 2023": "pedra"})

    elif ano == 2024:
        if "INDE 2024" in df.columns:
            df = df.rename(columns={"INDE 2024": "inde"})
        if "Pedra 2024" in df.columns:
            df = df.rename(columns={"Pedra 2024": "pedra"})

    return df

In [ ]:
df_2022 = padronizar_colunas(df_2022, 2022)
df_2023 = padronizar_colunas(df_2023, 2023)
df_2024 = padronizar_colunas(df_2024, 2024)

In [ ]:
colunas_interesse = [
    "RA", "Fase", "Turma", "ano", "nome",
    "ano_nascimento", "data_nascimento", "idade",
    "genero", "ano_ingresso", "instituicao_ensino",
    "pedra", "inde", "num_avaliacoes", "fase_ideal",
    "defasagem", "ian", "iaa", "ieg", "ips", "ipp",
    "ida", "ipv", "mat", "por", "ing"
]

In [ ]:
def filtrar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in colunas_interesse if c in df.columns]
    return df[cols].copy()

In [ ]:
df_2022 = filtrar_colunas(df_2022)
df_2023 = filtrar_colunas(df_2023)
df_2024 = filtrar_colunas(df_2024)

In [ ]:
df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)
print(df.shape)

(3030, 26)


In [ ]:
df.head()

,RA,Fase,Turma,ano,nome,ano_nascimento,idade,genero,ano_ingresso,instituicao_ensino,pedra,inde,num_avaliacoes,fase_ideal,defasagem,ian,iaa,ieg,ips,ida,ipv,mat,por,ing,data_nascimento,ipp
0,RA-1,7,A,2022,Aluno-1,2003.0,19,Menina,2016,Escola Pública,Quartzo,5.783,4.0,Fase 8 (Universitários),-1,5.0,8.3,4.1,5.6,4.0,7.278,2.7,3.5,6.0,NaN,NaN
1,RA-2,7,A,2022,Aluno-2,2005.0,17,Menina,2017,Rede Decisão,Ametista,7.055,4.0,Fase 7 (3º EM),0,10.0,8.8,5.2,6.3,6.8,6.778,6.3,4.5,9.7,NaN,NaN
2,RA-3,7,A,2022,Aluno-3,2005.0,17,Menina,2016,Rede Decisão,Ágata,6.591,4.0,Fase 7 (3º EM),0,10.0,0.0,7.9,5.6,5.6,7.556,5.8,4.0,6.9,NaN,NaN
3,RA-4,7,A,2022,Aluno-4,2005.0,17,Menino,2017,Rede Decisão,Quartzo,5.951,4.0,Fase 7 (3º EM),0,10.0,8.8,4.5,5.6,5.0,5.278,2.8,3.5,8.7,NaN,NaN
4,RA-5,7,A,2022,Aluno-5,2005.0,17,Menina,2016,Rede Decisão,Ametista,7.427,4.0,Fase 7 (3º EM),0,10.0,7.9,8.6,5.6,5.2,7.389,7.0,2.9,5.7,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Fase                3030 non-null   object 
 2   Turma               3030 non-null   object 
 3   ano                 3030 non-null   int64  
 4   nome                3030 non-null   object 
 5   ano_nascimento      860 non-null    float64
 6   idade               3030 non-null   object 
 7   genero              3030 non-null   object 
 8   ano_ingresso        3030 non-null   int64  
 9   instituicao_ensino  3029 non-null   object 
 10  pedra               2883 non-null   object 
 11  inde                2883 non-null   object 
 12  num_avaliacoes      2954 non-null   float64
 13  fase_ideal          3030 non-null   object 
 14  defasagem           3030 non-null   int64  
 15  ian                 3030 non-null   float64
 16  iaa   

In [ ]:
df = df.replace("INCLUIR", np.nan)

/tmp/ipykernel_7760/1144416605.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("INCLUIR", np.nan)


In [ ]:
df["data_nascimento"] = pd.to_datetime(df["data_nascimento"], errors="coerce")

df["ano_nascimento_from_data"] = df["data_nascimento"].dt.year

df["ano_nascimento"] = df["ano_nascimento"].fillna(df["ano_nascimento_from_data"])

df["ano_nascimento"] = df["ano_nascimento"].astype(int)

df = df.drop(columns=["ano_nascimento_from_data", "data_nascimento"])

In [ ]:
def corrigir_idade(valor):
    if pd.isna(valor):
        return np.nan

    # Caso seja datetime (Excel bug)
    if isinstance(valor, (pd.Timestamp, datetime)):
        return float(valor.day)

    # Caso seja string tipo data
    if isinstance(valor, str):
        try:
            dt = pd.to_datetime(valor, errors="raise")
            return float(dt.day)
        except:
            pass

    # Caso seja número normal
    try:
        return float(valor)
    except:
        return np.nan

In [ ]:
df["idade"] = df["idade"].apply(corrigir_idade)
df["idade"] = pd.to_numeric(df["idade"], errors="coerce")
df["idade"] = df["idade"].astype(int)

In [ ]:
df["num_avaliacoes"] = (pd.to_numeric(df["num_avaliacoes"], errors="coerce").fillna(0).astype(int))

In [ ]:
cols_numericas = [
    "ian", "iaa", "ieg", "ips", "ipp", "ida",
    "ipv", "mat", "por", "ing", "inde"
]

for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Fase                3030 non-null   object 
 2   Turma               3030 non-null   object 
 3   ano                 3030 non-null   int64  
 4   nome                3030 non-null   object 
 5   ano_nascimento      3030 non-null   int64  
 6   idade               3030 non-null   int64  
 7   genero              3030 non-null   object 
 8   ano_ingresso        3030 non-null   int64  
 9   instituicao_ensino  3029 non-null   object 
 10  pedra               2845 non-null   object 
 11  inde                2845 non-null   float64
 12  num_avaliacoes      3030 non-null   int64  
 13  fase_ideal          3030 non-null   object 
 14  defasagem           3030 non-null   int64  
 15  ian                 3030 non-null   float64
 16  iaa   

In [ ]:
df["genero"] = df["genero"].replace({
    "Menina": "Feminino",
    "Menino": "Masculino"
})

In [ ]:
df["pedra"] = df["pedra"].replace({"Agata": "Ágata"})

In [ ]:
map_pedra = {
    "Quartzo": 1,
    "Ágata": 2,
    "Ametista": 3,
    "Topázio": 4
}

df["pedra_ordem"] = df["pedra"].map(map_pedra)

In [ ]:
def extrair_fase_num(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()

    if valor == "ALFA":
        return 0

    match_fase = re.search(r"FASE\s*(\d+)", valor)
    if match_fase:
        return int(match_fase.group(1))

    match_codigo = re.match(r"(\d+)", valor)
    if match_codigo:
        return int(match_codigo.group(1))

    return np.nan

In [ ]:
df["Fase"] = df["Fase"].apply(extrair_fase_num)

In [ ]:
def extrair_turma_letra(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()

    match_alfa = re.search(r"ALFA\s+([A-Z])", valor)
    if match_alfa:
        return match_alfa.group(1)

    match_codigo = re.match(r"\d+([A-Z])", valor)
    if match_codigo:
        return match_codigo.group(1)

    if len(valor) == 1 and valor.isalpha():
        return valor

    if valor == "9":
        return "9"

    return np.nan

In [ ]:
df["Turma"] = df["Turma"].apply(extrair_turma_letra)

In [ ]:
def extrair_fase_ideal_num(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()

    if "ALFA" in valor:
        return 0

    match = re.search(r"FASE\s*(\d+)", valor)
    if match:
        return int(match.group(1))

    return np.nan

In [ ]:
df["fase_ideal"] = df["fase_ideal"].apply(extrair_fase_ideal_num)

In [ ]:
notas_cols = ["mat", "por", "ing"]

df["num_notas_disponiveis"] = df[notas_cols].notna().sum(axis=1)
df["media_notas_disponiveis"] = df[notas_cols].mean(axis=1, skipna=True)

In [ ]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)
    return valor

df["instituicao_ensino"] = df["instituicao_ensino"].apply(normalizar_texto)

def agrupar_instituicao(valor):
    if pd.isna(valor):
        return np.nan

    v = str(valor).lower()

    # 1. Concluiu / Formado
    if "concluiu" in v or "formado" in v:
        return "Concluiu EM"

    # 2. Universitário
    if "universitário" in v or "universitario" in v:
        return "Universitário"

    # 3. Pública
    if "pública" in v or "publica" in v:
        return "Pública"

    # 4. Privada com bolsa/parceria
    if "apadrinhamento" in v or "bolsa" in v or "parceira" in v:
        return "Privada com bolsa/apadrinhamento"

    # 5. Privada (inclui escolas específicas)
    if "privada" in v or "escola" in v or "rede" in v:
        return "Privada"

    return "Outros"

In [ ]:
df["instituicao_ensino"] = df["instituicao_ensino"].apply(agrupar_instituicao)

In [ ]:
df["ra_num"] = (
    df["RA"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

In [ ]:
df["ra_num"] = df["ra_num"].astype("Int64")

In [ ]:
df = df.sort_values(["ra_num", "ano"]).reset_index(drop=True)

In [ ]:
df = df.drop(columns=["ra_num"])

In [ ]:
df.to_csv("base_tratada.csv", index=False)